In [25]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import *
from pyspark.sql.window import Window


StatementMeta(, 71e913e4-f7db-4e0b-8a1e-35713398c265, 27, Finished, Available, Finished, False)

In [26]:
silver_path = "abfss://19509fa9-5985-4a20-a114-28961871f698@onelake.dfs.fabric.microsoft.com/65a2f421-9db7-4ec8-8fa6-39387d60b013/Tables/dbo/"

StatementMeta(, 71e913e4-f7db-4e0b-8a1e-35713398c265, 28, Finished, Available, Finished, False)

In [27]:
fact = spark.read.format("delta").load(
    f"{silver_path}fact_mandi_prices"
)
dim_c = spark.read.format("delta").load(
    f"{silver_path}dim_commodity"
)
dim_d = spark.read.format("delta").load(
    f"{silver_path}dim_date"
)
dim_l = spark.read.format("delta").load(
    f"{silver_path}dim_location"
)


StatementMeta(, 71e913e4-f7db-4e0b-8a1e-35713398c265, 29, Finished, Available, Finished, False)

In [28]:
# Working table with all readable columns
df = (
    fact
    .join(dim_c, "commodity_key")
    .join(dim_d, "date_key")
    .join(dim_l, "location_key")
    .select(
        "arrival_date", "year", "month",
        "state", "district", "market",
        "commodity", "variety", "grade",
        "min_price", "max_price", "modal_price"
    )
)

df.cache()
print("Working rows:", df.count())


StatementMeta(, 71e913e4-f7db-4e0b-8a1e-35713398c265, 30, Finished, Available, Finished, False)

Working rows: 58532


In [30]:
# Window specs — partitioned by commodity+market, ordered by date
w7  = Window.partitionBy("commodity", "variety", "market") \
             .orderBy("arrival_date") \
             .rowsBetween(-6, 0)   # 7-day rolling

w30 = Window.partitionBy("commodity", "variety", "market") \
             .orderBy("arrival_date") \
             .rowsBetween(-29, 0)  # 30-day rolling

volatility = (
    df
    .withColumn("stddev_7d",  round(stddev("modal_price").over(w7),  2))
    .withColumn("stddev_30d", round(stddev("modal_price").over(w30), 2))
    .withColumn("avg_7d",     round(avg("modal_price").over(w7),     2))
    .withColumn("avg_30d",    round(avg("modal_price").over(w30),    2))
    # Coefficient of variation = stddev / avg * 100
    # Gives relative volatility regardless of commodity price scale
    .withColumn("cv_7d",  round(
        when(col("avg_7d") > 0, col("stddev_7d") / col("avg_7d") * 100)
        .otherwise(lit(None)), 2))
    .withColumn("cv_30d", round(
        when(col("avg_30d") > 0, col("stddev_30d") / col("avg_30d") * 100)
        .otherwise(lit(None)), 2))
    # Classify volatility level for easy Power BI filtering
    .withColumn("volatility_band_7d",
        when(col("cv_7d") < 5,   lit("Low"))
        .when(col("cv_7d") < 15, lit("Medium"))
        .when(col("cv_7d") >= 15, lit("High"))
        .otherwise(lit(None)))
    .select(
        "arrival_date", "year", "month",
        "state", "district", "market",
        "commodity", "variety", "grade",
        "modal_price",
        "stddev_7d", "avg_7d", "cv_7d", "volatility_band_7d",
        "stddev_30d", "avg_30d", "cv_30d"
    )
)

volatility = volatility.withColumn(
    "date_commodity_key",
    concat_ws("_", col("arrival_date").cast("string"), col("commodity"))
)

volatility.write.format("delta").mode("overwrite") \
    .saveAsTable("gold_price_volatility")

print("Volatility rows:", volatility.count())
volatility.orderBy(col("cv_7d").desc()).show(10)

StatementMeta(, 71e913e4-f7db-4e0b-8a1e-35713398c265, 32, Finished, Available, Finished, False)

Volatility rows: 58532
+------------+----+-----+-------+---------+--------------------+--------------------+-------+-----+-----------+---------+-------+------+------------------+----------+-------+------+--------------------+
|arrival_date|year|month|  state| district|              market|           commodity|variety|grade|modal_price|stddev_7d| avg_7d| cv_7d|volatility_band_7d|stddev_30d|avg_30d|cv_30d|  date_commodity_key|
+------------+----+-----+-------+---------+--------------------+--------------------+-------+-----+-----------+---------+-------+------+------------------+----------+-------+------+--------------------+
|  2023-08-23|2023|    8|Gujarat|Ahmedabad|           Ahmedabad|    Mango (Raw-Ripe)|  Other|  FAQ|     7900.0|  2497.64|2321.43|107.59|              High|   1202.29|1886.67| 63.73|2023-08-23_Mango ...|
|  2025-12-01|2025|   12|Gujarat|Ahmedabad|      Ahmedabad Apmc|Rat Tail Radish(M...|  Other|  FAQ|     4500.0|  2722.36| 2575.0|105.72|              High|   2722.36

In [31]:
# Daily price spread per commodity
arbitrage = (
    df
    .groupBy("arrival_date", "year", "month", "state", "district", "commodity")
    .agg(
        max("modal_price").alias("max_modal_price"),
        min("modal_price").alias("min_modal_price"),
        avg("modal_price").alias("avg_modal_price"),
        count("*").alias("variety_count")
    )
    .withColumn("price_spread",
        round(col("max_modal_price") - col("min_modal_price"), 2))
    .withColumn("spread_pct",
        round(
            when(col("avg_modal_price") > 0,
                col("price_spread") / col("avg_modal_price") * 100)
            .otherwise(lit(None)), 2))
    # Only meaningful when more than one variety exists
    .withColumn("arbitrage_flag",
        when(
            (col("variety_count") > 1) & (col("spread_pct") > 20),
            lit("High spread")
        ).when(
            (col("variety_count") > 1) & (col("spread_pct") > 10),
            lit("Moderate spread")
        ).otherwise(lit("Low / single variety")))
    .orderBy("arrival_date", col("price_spread").desc())
)

arbitrage = arbitrage.withColumn(
    "date_commodity_key",
    concat_ws("_", col("arrival_date").cast("string"), col("commodity"))
)

arbitrage.write.format("delta").mode("overwrite") \
    .saveAsTable("gold_arbitrage_index")

print("Arbitrage rows:", arbitrage.count())
arbitrage.filter(col("spread_pct") > 20).show(10)

StatementMeta(, 71e913e4-f7db-4e0b-8a1e-35713398c265, 33, Finished, Available, Finished, False)

Arbitrage rows: 52867
+------------+----+-----+-------+---------+---------+---------------+---------------+---------------+-------------+------------+----------+--------------+------------------+
|arrival_date|year|month|  state| district|commodity|max_modal_price|min_modal_price|avg_modal_price|variety_count|price_spread|spread_pct|arbitrage_flag|date_commodity_key|
+------------+----+-----+-------+---------+---------+---------------+---------------+---------------+-------------+------------+----------+--------------+------------------+
|  2020-03-28|2020|    3|Gujarat|Ahmedabad|    Onion|         1850.0|         1500.0|         1675.0|            2|       350.0|      20.9|   High spread|  2020-03-28_Onion|
|  2020-06-08|2020|    6|Gujarat|Ahmedabad|    Mango|         5200.0|         4200.0|         4700.0|            4|      1000.0|     21.28|   High spread|  2020-06-08_Mango|
|  2020-06-10|2020|    6|Gujarat|Ahmedabad|    Mango|         5300.0|         4200.0|         4725.0|       

In [32]:
# Check volatility table
print("=== gold_price_volatility ===")
spark.table("gold_price_volatility") \
    .groupBy("volatility_band_7d").count() \
    .orderBy("volatility_band_7d").show()

# Most volatile commodities in Ahmedabad
spark.table("gold_price_volatility") \
    .groupBy("commodity") \
    .agg(avg("cv_7d").alias("avg_volatility")) \
    .orderBy(col("avg_volatility").desc()) \
    .show(10)

# Check arbitrage table
print("=== gold_arbitrage_index ===")
spark.table("gold_arbitrage_index") \
    .groupBy("arbitrage_flag").count().show()

# Top spread days
spark.table("gold_arbitrage_index") \
    .orderBy(col("spread_pct").desc()) \
    .show(10)

StatementMeta(, 71e913e4-f7db-4e0b-8a1e-35713398c265, 34, Finished, Available, Finished, False)

=== gold_price_volatility ===
+------------------+-----+
|volatility_band_7d|count|
+------------------+-----+
|              NULL|  205|
|              High|15568|
|               Low|14069|
|            Medium|28690|
+------------------+-----+

+--------------------+------------------+
|           commodity|    avg_volatility|
+--------------------+------------------+
|              Turnip|  28.1390909090909|
|Rat Tail Radish(M...| 26.93066666666666|
|Bajra(Pearl Mille...|            26.675|
|        Barley (Jau)|            26.075|
|               Soanf|             22.52|
|   Coriander(Leaves)|22.125454545454545|
|             Raddish| 20.10182212581346|
|Surat Beans (Papadi)|  19.7941162227603|
|       Methi(Leaves)| 19.34750000000001|
|               Tinda|19.305910828025496|
+--------------------+------------------+
only showing top 10 rows

=== gold_arbitrage_index ===
+--------------------+-----+
|      arbitrage_flag|count|
+--------------------+-----+
|         High spread| 